In [1]:
# --- Imports ---
import os
import numpy as np
import librosa
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from random import sample
import joblib

# --- Data augmentation functions ---
def time_stretch(audio, rate=1.1):
    return librosa.effects.time_stretch(y=audio, rate=rate)

def add_noise(audio, noise_factor=0.005):
    noise = np.random.randn(len(audio))
    augmented_audio = audio + noise_factor * noise
    return augmented_audio

# --- Feature extraction ---
def load_data(main_directory, files, augment=False, size=64, n_mfcc=40, n_mels=40):
    data = []
    labels = []
    for i, folder in sorted(enumerate(os.listdir(main_directory))):
        folder_path = os.path.join(main_directory, folder)
        if os.path.isdir(folder_path):
            for file in os.listdir(folder_path):
                if file.endswith(".wav") and file in files:
                    file_path = os.path.join(folder_path, file)
                    audio, sr = librosa.load(file_path, sr=44000)

                    if augment:
                        if np.random.random() > 0.5:
                            audio = time_stretch(audio, rate=np.random.uniform(0.8, 1.2))
                        if np.random.random() > 0.5:
                            audio = add_noise(audio, noise_factor=np.random.uniform(0.001, 0.01))

                    # --- MFCC feature ---
                    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
                    mfccs = resize(np.expand_dims(mfccs, axis=-1), (size, size))

                    # --- Mel Spectrogram feature ---
                    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
                    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
                    mel_spec_db = resize(np.expand_dims(mel_spec_db, axis=-1), (size, size))

                    # --- Concatenate MFCC and Mel spectrogram ---
                    combined_feature = np.concatenate((mfccs, mel_spec_db), axis=-1)

                    data.append(combined_feature)
                    labels.append(i)

    return np.array(data), np.array(labels)

# --- Paths ---
main_directory = "/content/drive/MyDrive/Cat_Dog_Detection/mydata"

# --- Select files ---
files = os.listdir(os.path.join(main_directory, "cat"))
files.extend(sample(
    os.listdir(os.path.join(main_directory, "other")),
    len(os.listdir(os.path.join(main_directory, "cat")))
))

# --- Split files ---
train_sample, test_sample = train_test_split(files, test_size=0.1, random_state=42)
train_sample, val_sample = train_test_split(train_sample, test_size=0.1, random_state=42)

print(f"Train: {len(train_sample)}, Val: {len(val_sample)}, Test: {len(test_sample)}")

# --- Parameters to test ---
n_mfcc_list =[6, 12, 20, 40, 80]
n_mels_list = [6,12,20, 40, 80]
rf_estimators_list = [10,20,50,100,200, 500, 1000]
knn_neighbors_list = [5, 10, 15]
size_list = [16,32,64,128,256]
augment_options = [False]
max_acc = 0
model_name = None

# --- Start experiments ---
for augment in augment_options:
    for n_mfcc in n_mfcc_list:
        for n_mels in n_mels_list:
          for size in size_list:
              print(f"\n=== Experiment: Augment={augment}, n_mfcc={n_mfcc}, n_mels={n_mels} ===, image_size==={size}")

              # Load data
              x_train, y_train = load_data(main_directory, train_sample, augment=augment, n_mfcc=n_mfcc, n_mels=n_mels,size=size)
              x_val, y_val = load_data(main_directory, val_sample, augment=False, n_mfcc=n_mfcc, n_mels=n_mels,size=size)
              x_test, y_test = load_data(main_directory, test_sample, augment=False, n_mfcc=n_mfcc, n_mels=n_mels,size=size)

              # Flatten features
              x_train_flat = x_train.reshape(x_train.shape[0], -1)
              x_val_flat = x_val.reshape(x_val.shape[0], -1)
              x_test_flat = x_test.reshape(x_test.shape[0], -1)

              # --- Random Forest experiments ---
              for n_estimators in rf_estimators_list:
                  rf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
                  rf.fit(x_train_flat, y_train)
                  y_pred_rf = rf.predict(x_test_flat)
                  test_acc_rf = accuracy_score(y_test, y_pred_rf)
                  print(f"Random Forest (n_estimators={n_estimators}): Test Accuracy = {test_acc_rf:.4f}")
                  if test_acc_rf > max_acc:
                      max_acc = test_acc_rf
                      model_name = f"rf_{n_estimators}_mf{n_mfcc}_ml{n_mels}_ims_{size}_acc_ {test_acc_rf:.3f}.pkl"
                      joblib.dump(rf, os.path.join(main_directory,model_name))
                      print("New Models Saved")




              # --- Optional: SVM (keep fixed) ---
              svm = SVC(kernel='linear', probability=True)
              svm.fit(x_train_flat, y_train)
              y_pred_svm = svm.predict(x_test_flat)
              test_acc_svm = accuracy_score(y_test, y_pred_svm)
              print(f"SVM: Test Accuracy = {test_acc_svm:.4f}")
              if test_acc_svm > max_acc:
                  max_acc = test_acc_svm
                  model_name = f"sv_mf{n_mfcc}_ml{n_mels}_ims_{size}_acc_ {test_acc_svm:.3f}.pkl"
                  joblib.dump(svm, os.path.join(main_directory,model_name))
                  print("New Models Saved")



Train: 395, Val: 44, Test: 49

=== Experiment: Augment=False, n_mfcc=6, n_mels=6 ===, image_size===16
Random Forest (n_estimators=10): Test Accuracy = 0.7959
New Models Saved
Random Forest (n_estimators=50): Test Accuracy = 0.8163
New Models Saved
Random Forest (n_estimators=100): Test Accuracy = 0.7959
Random Forest (n_estimators=200): Test Accuracy = 0.8163
Random Forest (n_estimators=500): Test Accuracy = 0.8163
Random Forest (n_estimators=1000): Test Accuracy = 0.8163
SVM: Test Accuracy = 0.6939

=== Experiment: Augment=False, n_mfcc=6, n_mels=6 ===, image_size===32
Random Forest (n_estimators=10): Test Accuracy = 0.8980
New Models Saved
Random Forest (n_estimators=50): Test Accuracy = 0.8367
Random Forest (n_estimators=100): Test Accuracy = 0.8571
Random Forest (n_estimators=200): Test Accuracy = 0.7959
Random Forest (n_estimators=500): Test Accuracy = 0.8163
Random Forest (n_estimators=1000): Test Accuracy = 0.8163
SVM: Test Accuracy = 0.7347

=== Experiment: Augment=False, n_mfc